In [4]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')         # Fix: prevents display error on Windows/PyCharm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

os.makedirs('plots', exist_ok=True)

# Load dataset
df = pd.read_csv('../data/creditcard.csv')

print("DATASET OVERVIEW")
print("=" * 50)
print(f"Shape:          {df.shape}")
print(f"Columns:        {list(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")
print()
print("CLASS DISTRIBUTION:")
counts = df['Class'].value_counts()
print(f"  Legitimate (0): {counts[0]:,}  ({counts[0]/len(df)*100:.2f}%)")
print(f"  Fraud      (1): {counts[1]:,}   ({counts[1]/len(df)*100:.3f}%)")
print(f"  Imbalance:      {counts[0]/counts[1]:.0f}:1 ratio")
print()
print(df.describe().round(2))

DATASET OVERVIEW
Shape:          (20000, 31)
Columns:        ['Time', 'Amount', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class']
Missing values: 0

CLASS DISTRIBUTION:
  Legitimate (0): 19,968  (99.84%)
  Fraud      (1): 32   (0.160%)
  Imbalance:      624:1 ratio

            Time    Amount        V1        V2        V3        V4        V5  \
count   20000.00  20000.00  20000.00  20000.00  20000.00  20000.00  20000.00   
mean    86282.77     80.40      0.00      0.01     -0.01     -0.01      0.01   
std     50066.66     80.34      1.00      1.00      1.01      0.99      1.00   
min         5.00      0.00     -4.19     -3.94     -3.91     -3.67     -3.85   
25%     42712.25     23.54     -0.68     -0.67     -0.69     -0.67     -0.65   
50%     86705.00     56.25     -0.00      0.02      0.01     -0.01      0.00   
75%    130168.25    109.97 

In [5]:
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

print("AMOUNT ANALYSIS:")
print(f"  Legitimate — Mean: ${legit['Amount'].mean():.2f}  Median: ${legit['Amount'].median():.2f}")
print(f"  Fraud      — Mean: ${fraud['Amount'].mean():.2f}  Median: ${fraud['Amount'].median():.2f}")

# Amount bracket analysis
df['Amount_Bracket'] = pd.cut(
    df['Amount'],
    bins=[0, 25, 100, 500, 5000, 100000],
    labels=['Micro', 'Small', 'Medium', 'Large', 'VeryLarge']
)
fraud_by_amount = df.groupby('Amount_Bracket', observed=True)['Class'].agg(['sum','count'])
fraud_by_amount['fraud_rate_pct'] = (
    fraud_by_amount['sum'] / fraud_by_amount['count'] * 100
).round(2)
print()
print("FRAUD RATE BY AMOUNT BRACKET:")
print(fraud_by_amount.to_string())

AMOUNT ANALYSIS:
  Legitimate — Mean: $80.46  Median: $56.27
  Fraud      — Mean: $47.08  Median: $34.27

FRAUD RATE BY AMOUNT BRACKET:
                sum  count  fraud_rate_pct
Amount_Bracket                            
Micro            12   5248            0.23
Small            16   9048            0.18
Medium            4   5662            0.07
Large             0     40            0.00


In [7]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Fraud Detection EDA', fontsize=14, fontweight='bold')

# Chart 1: Class distribution
counts = df['Class'].value_counts().sort_index()
bars = axes[0,0].bar(['Legitimate (0)', 'Fraud (1)'],
                      counts.values,
                      color=['#3B82F6', '#EF4444'],
                      edgecolor='black', alpha=0.85)
axes[0,0].set_title('Class Distribution', fontweight='bold')
for bar, val in zip(bars, counts.values):
    axes[0,0].text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+1000,
                    f'{val:,}', ha='center', fontweight='bold')

# Chart 2: Amount distribution (log scale)
axes[0,1].hist(legit['Amount'].clip(upper=500), bins=60,
               alpha=0.6, color='green', label='Legitimate', density=True)
axes[0,1].hist(fraud['Amount'].clip(upper=500), bins=60,
               alpha=0.7, color='red', label='Fraud', density=True)
axes[0,1].set_title('Amount: Fraud vs Legitimate', fontweight='bold')
axes[0,1].set_xlabel('Amount ($)')
axes[0,1].legend()

# Chart 3: Time pattern (hours)
df['Hour'] = (df['Time'] // 3600) % 24
fraud_by_hour = df[df['Class']==1].groupby('Hour').size()
axes[1,0].bar(fraud_by_hour.index, fraud_by_hour.values,
               color='red', edgecolor='black', alpha=0.75)
axes[1,0].set_title('Fraud Count by Hour of Day', fontweight='bold')
axes[1,0].set_xlabel('Hour')
axes[1,0].set_ylabel('Fraud Count')

# Chart 4: V14 distribution (top SHAP feature)
axes[1,1].hist(legit['V14'].clip(-10, 10), bins=60,
               alpha=0.6, color='green', label='Legitimate', density=True)
axes[1,1].hist(fraud['V14'].clip(-10, 10), bins=60,
               alpha=0.7, color='red', label='Fraud', density=True)
axes[1,1].set_title('V14 — Top Fraud Separator', fontweight='bold')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('../plots/fraud_eda.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: plots/fraud_eda.png")

print()
print("YOUR 3 INTERVIEW INSIGHTS:")
fraud_micro_pct = df[df['Amount'] < 25]['Class'].mean() * 100
print(f"1. Micro transactions (<$25) have {fraud_micro_pct:.2f}% fraud rate (card testing)")
print(f"2. Fraud median amount ${fraud['Amount'].median():.2f} vs Legit ${legit['Amount'].median():.2f}")
print(f"3. V14 is the strongest single fraud predictor (from SHAP analysis)")

Saved: plots/fraud_eda.png

YOUR 3 INTERVIEW INSIGHTS:
1. Micro transactions (<$25) have 0.23% fraud rate (card testing)
2. Fraud median amount $34.27 vs Legit $56.27
3. V14 is the strongest single fraud predictor (from SHAP analysis)
